In [ ]:
# Sección 1: Configuración del entorno en Colab

# 1. Instalar PySpark
!pip install pyspark
from pyspark.sql import SparkSession

# 3. Crear SparkSession configurando el nombre de la app
spark = SparkSession.builder.appName("BigDataLab").getOrCreate()

print(f"\nSparkSession configurada exitosamente. Versión de Spark: {spark.version}")


SparkSession configurada exitosamente. Versión de Spark: 4.0.2


In [5]:
# Sección 3: Verificación

# 1. Crear un pequeño conjunto de datos de prueba
datos_prueba = [
    "Spark es muy rapido",
    "Procesamiento de Big Data",
    "Spark usa memoria en vez de disco para ser rapido"
]

# 2. Crear un RDD (Resilient Distributed Dataset) a partir de la lista
rdd_prueba = spark.sparkContext.parallelize(datos_prueba)

# 3. Ejecutar una operación simple de MapReduce (Conteo de palabras básico)
# - flatMap: divide cada frase en palabras
# - map: convierte la palabra en una tupla (palabra, 1)
# - reduceByKey: agrupa las palabras iguales y suma sus valores
conteo_palabras = rdd_prueba.flatMap(lambda frase: frase.split(" ")) \
                            .map(lambda palabra: (palabra, 1)) \
                            .reduceByKey(lambda a, b: a + b)

# 4. Imprimir resultado usando collect() para traer los datos del RDD
print("Resultado del conteo de palabras (Verificación):\n")
for resultado in conteo_palabras.collect():
    print(f"{resultado[0]}: {resultado[1]}")

Resultado del conteo de palabras (Verificación):

es: 1
muy: 1
Procesamiento: 1
Big: 1
memoria: 1
vez: 1
para: 1
ser: 1
Spark: 2
rapido: 2
de: 2
Data: 1
usa: 1
en: 1
disco: 1


## 2. Carga y Procesamiento de Dataset

In [ ]:
# Sección 4: Carga del dataset desde Google Drive

# 1. Conectar Google Drive con Colab para poder leer el archivo
from google.colab import drive
drive.mount('/content/drive')

# 2. Definir ruta. (G:\Mi unidad equi|vale a /content/drive/MyDrive en Colab)
ruta_archivo = "/content/drive/MyDrive/Colab Notebooks/Big data files/amazon_review.csv"

# 3. Leer el dataset con Spark
try:
    df_resenas = spark.read.csv(ruta_archivo, header=True, inferSchema=True)
    
    # 4. Mostrar el schema
    print("\n--- Schema del dataset ---")
    df_resenas.printSchema()
    
    print("\n--- Primeras filas ---")
    df_resenas.show(5, truncate=True)
except Exception as e:
    print("Ocurrió un error al cargar el archivo. Verifica la ruta:\n", e)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).

--- Schema del dataset ---
root
 |-- id: string (nullable = true)
 |-- asins: string (nullable = true)
 |-- brand: string (nullable = true)
 |-- categories: string (nullable = true)
 |-- colors: string (nullable = true)
 |-- dateAdded: timestamp (nullable = true)
 |-- dateUpdated: timestamp (nullable = true)
 |-- dimension: string (nullable = true)
 |-- ean: double (nullable = true)
 |-- keys: string (nullable = true)
 |-- manufacturer: string (nullable = true)
 |-- manufacturerNumber: string (nullable = true)
 |-- name: string (nullable = true)
 |-- prices: string (nullable = true)
 |-- reviewsDate: string (nullable = true)
 |-- reviewsDoRecommend: string (nullable = true)
 |-- reviewsNumHelpful: string (nullable = true)
 |-- reviewsRating: string (nullable = true)
 |-- reviewsSourceURLs: string (nullable = true)
 |-- reviewsText: string (nullable = true)
 

In [ ]:
# Sección 5: Selección de datos y preprocesamiento básico
import re

nombre_columna_texto = "`reviewsText`"  

try:
    # Seleccionar la columna de texto y filtrar los registros que sean nulos
    df_texto = df_resenas.select(nombre_columna_texto).dropna()

    # Pasamos el DataFrame un RDD y extraemos el texto crudo del objeto Row
    rdd_crudo = df_texto.rdd.map(lambda fila: str(fila[0]))

    # Definir la función de limpieza
    def preprocesar_texto(texto):
        texto = texto.lower()
        texto = re.sub(r'[^a-z\s]', '', texto)
        return texto
    rdd_limpio = rdd_crudo.map(preprocesar_texto)
    print("Datos seleccionados y listos en un RDD para aplicar MapReduce.")
    
except Exception as e:
    print("Error en el preprocesamiento. Revisa que el nombre de la columna sea correcto:\n", e)

Datos seleccionados y listos en un RDD para aplicar MapReduce.


In [ ]:
# Implementación tipo MapReduce y Salida

#  Aplicación del flujo distribuido
# - flatMap: Separamos cada cadena por sus espacios para tokenizar.
# - filter: Por si quedaron cadenas vacías, las filtramos.
# - map: Transformamos cada token (palabra) a una tupla de valor 1 -> (palabra, 1)
# - reduceByKey: Agrupamos todas las palabras idénticas y sumamos sus unos.
rdd_conteo = rdd_limpio.flatMap(lambda texto: texto.split(" ")) \
                       .filter(lambda token: token.strip() != "") \
                       .map(lambda palabra: (palabra, 1)) \
                       .reduceByKey(lambda a, b: a + b)

# Cómputo Final y Salida Top 20
# Usamos sortBy sobre el segundo elemento x[1] de las tuplas resultantes en orden inverso (ascendent=False)
top_20 = rdd_conteo.sortBy(lambda x: x[1], ascending=False).take(20)

print("\n=== TOP 20 Palabras Más Frecuentes en las Reseñas ===")
for posicion, (palabra, frecuencia) in enumerate(top_20, 1):
    print(f"{posicion:02d}. '{palabra}' -> {frecuencia}")


=== TOP 20 Palabras Más Frecuentes en las Reseñas ===
01. 'tz' -> 1167
02. 'issalefalse' -> 252
03. 'shippingfree' -> 63
04. 'shipping' -> 63
05. 'merchantamazoncom' -> 60
06. 'sourceurlshttpswwwamazoncomgpproductbypmrefcondpavailkinkw' -> 32
07. 'sourceurlshttpwwwamazoncomkindlepaperwhiteresolutiondisplaybuiltdpbjggowu' -> 12
08. 'sourceurlshttpswwwamazoncomkindleoasisereaderleatherchargingdpbzsgpgreflpsamazondevicesieutfqidsr' -> 4
09. 'sourceurlshttpwwwamazoncomkindlepaperwhiteereaderdpbawhm' -> 4
10. 'sourceurlshttpwwwamazoncomfirehdxdisplaywifidpbhcnfee' -> 3


## 3. Análisis de Particiones y Balanceo de Carga

In [ ]:
# Sección 7: Medición de tiempos de ejecución por cada partición
import time

# Guardar el RDD en la memoria caché usando persist() / cache() para optimizar tiempo
# y se re-ejecute el Lowercase y el Regex explícitamente en cada iteración del experimento,
# aislando así el tiempo únicamente del proceso de re/particionado y MapReduce.
rdd_limpio.cache()

# Forzamos una acción tipo Count para materializar la estructura en el caché de Spark:
print(f"Materializando caché. Registros listos para test: {rdd_limpio.count()}\n")

# Lista de configuraciones de partición
particiones_experimento = [2, 4, 8, 16]
resultados = []

print("Iniciando evaluación comparativa de tiempo y particiones...")

# Proceso iterativo y recolección de tiempos
for n_particiones in particiones_experimento:
    
    # Re-particionado de los datos en el RDD
    rdd_particionado = rdd_limpio.repartition(n_particiones)
    
    start = time.time()
    
    # Ejecutamos el pipeline del MapReduce evaluado en la partición particular
    # Nota: usamos ".count()" obligatoriamente al final como "Acción" para que spark ejecute la transformación
    rdd_particionado.flatMap(lambda texto: texto.split(" ")) \
                    .filter(lambda token: token.strip() != "") \
                    .map(lambda palabra: (palabra, 1)) \
                    .reduceByKey(lambda a, b: a + b) \
                    .count() 

    end = time.time()
    tiempo_transcurrido = end - start
    
    # Agregamos a lista de resultados en memoria
    resultados.append((n_particiones, tiempo_transcurrido))
    print(f"[-] Finalizado test para {n_particiones} particiones. | {tiempo_transcurrido:.2f} segs.")

print("\n=======================================")
print("           RESULTADOS FINALES          ")
print("=======================================")
print(f"{str('Particiones'):<15} | {str('Tiempo (segundos)'):<20}")
print("----------------|----------------------")
for particion, tiempo in resultados:
    print(f"{str(particion):<15} | {tiempo:.4f} seg")
print("=======================================")

Materializando caché. Registros listos para test: 1597

Iniciando evaluación comparativa de tiempo y particiones...
[-] Finalizado test para 2 particiones. | 1.24 segs.
[-] Finalizado test para 4 particiones. | 1.96 segs.
[-] Finalizado test para 8 particiones. | 3.21 segs.
[-] Finalizado test para 16 particiones. | 7.38 segs.

           RESULTADOS FINALES          
Particiones     | Tiempo (segundos)   
----------------|----------------------
2               | 1.2413 seg
4               | 1.9578 seg
8               | 3.2060 seg
16              | 7.3844 seg


## 4. Consultas Distribuidas con API DataFrames

In [ ]:
# Exploración usando el API de DataFrames de Spark
# Anteriormente se utilizo RDD Resilient Distributed Datasets (bajo nivel), ahora se utilizara DataFrames (alto nivel y mayor optimización)

from pyspark.sql.functions import col, count, desc

print("=== Consultas Analíticas con DataFrames ===\n")

# Distribución de Calificaciones (Ratings): Agrupamos por la columna 'reviewsRating'
print("1. ¿Cuántas reseñas hay por calificación (Rating)?")
df_resenas.groupBy("reviewsRating") \
          .agg(count("*").alias("Total")) \
          .orderBy(desc("Total")) \
          .show()

# Top Marcas más comentadas
print("\n2. Top 5 Marcas con mayor número de reseñas:")
df_resenas.groupBy("brand") \
          .agg(count("*").alias("Total_Resenas")) \
          .orderBy(desc("Total_Resenas")) \
          .show(5, truncate=False)

# Filtrar reseñas productivas: Reseñas donde los usuarios recomiendan un producto
print("\n3. Productos recomendados por los usuarios (Muestra de 5):")
df_resenas.filter(col("reviewsDoRecommend") == "true") \
          .select("name", "reviewsRating", "reviewsDoRecommend") \
          .show(5, truncate=False)

=== Consultas Analíticas con DataFrames ===

1. ¿Cuántas reseñas hay por calificación (Rating)?
+--------------------+-----+
|       reviewsRating|Total|
+--------------------+-----+
|""dateSeen"":[""2...|  542|
|""dateAdded"":""2...|  166|
|""dateAdded"":""2...|  133|
|""dateSeen"":[""2...|   87|
|""dateSeen"":[""2...|   70|
|""dateSeen"":[""2...|   43|
|""dateSeen"":[""2...|   41|
|""dateAdded"":""2...|   38|
|""dateSeen"":[""2...|   32|
|""dateSeen"":[""2...|   27|
|""dateSeen"":[""2...|   23|
|""dateSeen"":[""2...|   20|
|""dateSeen"":[""2...|   19|
|""dateSeen"":[""2...|   18|
|""dateSeen"":[""2...|   17|
|""dateSeen"":[""2...|   13|
|""dateSeen"":[""2...|   13|
|""dateSeen"":[""2...|   12|
|""dateSeen"":[""2...|   12|
|""dateSeen"":[""2...|   12|
+--------------------+-----+
only showing top 20 rows

2. Top 5 Marcas con mayor número de reseñas:
+------+-------------+
|brand |Total_Resenas|
+------+-------------+
|Amazon|1585         |
|Moshi |12           |
+------+-------------+